# GéoMarketing IDF — J2 : Construire les indicateurs démographiques d’Île-de-France

Ce notebook transforme le fichier produit au J1 en un tableau démographique simple et exploitable pour GeoMarketing IDF.

Il calcule notamment :

- la population totale ;
- la population et la part des moins de 30 ans ;
- la part des 15–29 ans et des 30–44 ans ;
- la population et la part des 60 ans ou plus ;
- l’évolution de la population entre 2016 et 2022 ;
- le taux annuel moyen d’évolution entre 2016 et 2022.

> Le fichier `population_idf_2022.csv` du J1 n’est jamais modifié. Exécute les cellules dans l’ordre avec **Shift + Entrée**.

## 1. Configuration

Le notebook recherche automatiquement `GeoMarketing_IDF` dans OneDrive. Si nécessaire, renseigne seulement `RACINE_PROJET_MANUELLE`.

In [1]:
from pathlib import Path
import os

# Correction préventive de variables de threads Windows invalides.
for variable_threads in ("NUMEXPR_NUM_THREADS", "OMP_NUM_THREADS"):
    valeur = os.environ.get(variable_threads)
    if valeur is not None:
        valeur_nettoyee = valeur.strip()
        valeur_valide = (
            valeur_nettoyee.isdecimal() and int(valeur_nettoyee) >= 1
        )
        if not valeur_valide:
            print(
                f"Variable invalide supprimée : {variable_threads}={valeur!r}"
            )
            os.environ.pop(variable_threads, None)

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

# Exemple si la détection automatique ne fonctionne pas :
# RACINE_PROJET_MANUELLE = r"C:\\Users\\Kassim\\OneDrive\\GeoMarketing_IDF"
RACINE_PROJET_MANUELLE = None

if RACINE_PROJET_MANUELLE:
    RACINE_PROJET = Path(RACINE_PROJET_MANUELLE)
else:
    candidats_onedrive = [
        os.getenv("OneDrive"),
        os.getenv("OneDriveConsumer"),
        os.getenv("OneDriveCommercial"),
        str(Path.home() / "OneDrive"),
    ]
    candidats_onedrive = [Path(p) for p in candidats_onedrive if p]
    racines_possibles = [p / "GeoMarketing_IDF" for p in candidats_onedrive]
    RACINE_PROJET = next((p for p in racines_possibles if p.exists()), None)

    if RACINE_PROJET is None:
        chemins_testes = "\n".join(f"- {p}" for p in racines_possibles)
        raise FileNotFoundError(
            "Le dossier GeoMarketing_IDF est introuvable. "
            "Renseigne RACINE_PROJET_MANUELLE.\n"
            f"Chemins testés :\n{chemins_testes}"
        )

DOSSIER_DATA = RACINE_PROJET / "data"
DOSSIER_PROCESSED_INSEE = DOSSIER_DATA / "processed" / "insee"

print(f"Racine du projet : {RACINE_PROJET}")
print(f"Données traitées Insee : {DOSSIER_PROCESSED_INSEE}")

Variable invalide supprimée : OMP_NUM_THREADS='A'
Racine du projet : C:\Users\almou\OneDrive\GeoMarketing_IDF
Données traitées Insee : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee


## 2. Repérage du fichier produit au J1

In [2]:
FICHIER_ENTREE = DOSSIER_PROCESSED_INSEE / "population_idf_2022.csv"

if not FICHIER_ENTREE.exists():
    fichiers_trouves = list(DOSSIER_DATA.rglob("population_idf_2022.csv"))

    if len(fichiers_trouves) == 1:
        FICHIER_ENTREE = fichiers_trouves[0]
    elif not fichiers_trouves:
        raise FileNotFoundError(
            "population_idf_2022.csv est introuvable. "
            "Termine d’abord le notebook J1."
        )
    else:
        liste = "\n".join(f"- {p}" for p in fichiers_trouves)
        raise RuntimeError(
            f"Plusieurs copies de population_idf_2022.csv ont été trouvées :\n{liste}"
        )

FICHIER_SORTIE = DOSSIER_PROCESSED_INSEE / "indicateurs_demographiques_idf_2022.csv"

print(f"Fichier d’entrée : {FICHIER_ENTREE}")
print(f"Fichier de sortie : {FICHIER_SORTIE}")

Fichier d’entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\population_idf_2022.csv
Fichier de sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\indicateurs_demographiques_idf_2022.csv


## 3. Chargement du fichier J1

Toutes les colonnes sont d’abord lues comme du texte. Les variables démographiques utiles sont ensuite converties explicitement en nombres.

In [3]:
def detecter_separateur(fichier, encodage):
    with fichier.open("r", encoding=encodage) as f:
        premiere_ligne = f.readline()

    separateurs = [";", ",", "\t"]
    comptes = {sep: premiere_ligne.count(sep) for sep in separateurs}
    separateur = max(comptes, key=comptes.get)

    if comptes[separateur] == 0:
        raise ValueError(f"Séparateur impossible à détecter dans {fichier}")

    return separateur


def lire_csv(fichier):
    derniere_erreur = None

    for encodage in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            separateur = detecter_separateur(fichier, encodage)
            tableau = pd.read_csv(
                fichier,
                sep=separateur,
                encoding=encodage,
                dtype=str,
                low_memory=False,
            )
            print(
                f"Séparateur={repr(separateur)}, encodage={encodage}, "
                f"lignes={len(tableau):,}, colonnes={len(tableau.columns)}"
            )
            return tableau
        except (UnicodeDecodeError, pd.errors.ParserError) as erreur:
            derniere_erreur = erreur

    raise RuntimeError(f"Impossible de lire {fichier}") from derniere_erreur


source = lire_csv(FICHIER_ENTREE)
display(source.head())

Séparateur=';', encodage=utf-8-sig, lignes=1,266, colonnes=316


,CODGEO,NOM_COMMUNE,DEP,REG,P22_POP,P22_POP0014,P22_POP1529,P22_POP3044,P22_POP4559,P22_POP6074,P22_POP7589,P22_POP90P,P22_POPH,P22_H0014,P22_H1529,P22_H3044,P22_H4559,P22_H6074,P22_H7589,P22_H90P,...,C11_POP1524_CS7,C11_POP1524_CS8,C11_POP2554,C11_POP2554_CS1,C11_POP2554_CS2,C11_POP2554_CS3,C11_POP2554_CS4,C11_POP2554_CS5,C11_POP2554_CS6,C11_POP2554_CS7,C11_POP2554_CS8,C11_POP55P,C11_POP55P_CS1,C11_POP55P_CS2,C11_POP55P_CS3,C11_POP55P_CS4,C11_POP55P_CS5,C11_POP55P_CS6,C11_POP55P_CS7,C11_POP55P_CS8
0,75056,Paris,75,11,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,993833.659485844,139317.915204446,236008.325130779,226433.119611239,187488.270453186,136034.826971534,61826.6259934308,6724.57612122827,...,0.0,194574.334475504,1034662.78913126,719.191097949181,43442.977193228,425439.137634267,221138.58199289,177045.764906102,70946.0461682061,2331.24093979136,93599.8491988282,587175.121883931,469.823483841753,16803.3522836906,82499.0101536924,37952.3322087192,36925.0108628204,15082.5898307056,346427.090087952,51015.9129725086
1,77001,Achères-la-Forêt,77,11,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,570.012381953201,89.5927550033545,77.1938457519072,89.9614067886164,164.721962171179,100.262186702676,41.9512254040055,6.32900013146226,...,0.0,107.130434782609,436.45732689211,11.9033816425121,43.645732689211,119.033816425121,91.2592592592593,79.3558776167472,67.4524959742351,3.96779388083736,19.8389694041868,365.037037037037,0.0,7.93558776167472,23.8067632850242,23.8067632850242,23.8067632850242,7.93558776167472,249.971014492754,27.7745571658615
2,77002,Amillis,77,11,821.0,127.936213516012,122.351987081097,146.913593456989,151.60336129515,188.399316742963,59.3206016148823,24.4749262929075,399.085369953582,65.0401461858607,59.589684490348,68.102705495937,86.3412645210653,90.6468098177631,27.1927000917768,2.17205935083134,...,0.0,42.5853159796658,303.301725413489,0.0,15.4855694471512,27.0997465325146,77.427847235756,123.88455557721,42.5853159796658,0.0,16.8186906411922,252.889789952447,3.8713923617878,3.8713923617878,3.8713923617878,19.356961808939,7.7427847235756,19.356961808939,183.466938032821,11.3519664928086
3,77003,Amponville,77,11,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0,172.0,35.0,16.0,34.0,42.0,31.0,14.0,0.0,...,0.0,44.0,172.0,4.0,24.0,24.0,40.0,48.0,24.0,0.0,8.0,80.0,0.0,4.0,20.0,8.0,12.0,0.0,32.0,4.0
4,77004,Andrezel,77,11,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0,160.0,41.0,15.0,38.0,29.0,31.0,6.0,0.0,...,0.0,20.0,112.0,12.0,8.0,28.0,24.0,16.0,16.0,0.0,8.0,116.0,4.0,12.0,12.0,0.0,8.0,4.0,72.0,4.0


## 4. Sélection et conversion des variables utiles

In [5]:
colonnes_identification = ["CODGEO", "NOM_COMMUNE", "DEP", "REG"]
variables_demographiques = [
    "P11_POP",
    "P16_POP",
    "P22_POP",
    "P22_POP0014",
    "P22_POP1529",
    "P22_POP3044",
    "P22_POP4559",
    "P22_POP6074",
    "P22_POP7589",
    "P22_POP90P",
]
colonnes_requises = colonnes_identification + variables_demographiques
colonnes_manquantes = sorted(set(colonnes_requises) - set(source.columns))

if colonnes_manquantes:
    raise ValueError(f"Colonnes requises absentes : {colonnes_manquantes}")

df = source[colonnes_requises].copy()

df["CODGEO"] = df["CODGEO"].astype("string").str.strip().str.zfill(5)
df["NOM_COMMUNE"] = df["NOM_COMMUNE"].astype("string").str.strip()
df["DEP"] = df["DEP"].astype("string").str.strip()
df["REG"] = df["REG"].astype("string").str.strip().str.zfill(2)

def convertir_en_nombre(serie):
    texte = (
        serie.astype("string")
        .str.strip()
        .str.replace("\u202f", "", regex=False)
        .str.replace(" " , "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(texte, errors="coerce")

for colonne in variables_demographiques:
    valeurs_originales = df[colonne].copy()
    df[colonne] = convertir_en_nombre(df[colonne])

    conversion_impossible = (
        valeurs_originales.notna()
        & valeurs_originales.astype("string").str.strip().ne("")
        & df[colonne].isna()
    )

    if conversion_impossible.any():
        exemples = valeurs_originales.loc[conversion_impossible].unique()[:10]
        raise ValueError(
            f"Valeurs non numériques dans {colonne} : {exemples.tolist()}"
        )

print("Variables sélectionnées et converties.")
display(df.head())

Variables sélectionnées et converties.


,CODGEO,NOM_COMMUNE,DEP,REG,P11_POP,P16_POP,P22_POP,P22_POP0014,P22_POP1529,P22_POP3044,P22_POP4559,P22_POP6074,P22_POP7589,P22_POP90P
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508,513386.756029,457511.658564,387099.21535,302819.494109,154040.332162,24566.982278
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135095,159.522993,189.869837,347.331906,204.113784,89.442063,10.584321
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936214,122.351987,146.913593,151.603361,188.399317,59.320602,24.474926
3,77003,Amponville,77,11,388.0,351.0,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0
4,77004,Andrezel,77,11,299.0,285.0,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0


## 5. Contrôles du fichier source

In [6]:
if df["CODGEO"].isna().any():
    raise ValueError("Des CODGEO sont manquants.")

doublons = df.loc[df["CODGEO"].duplicated(keep=False)]
if not doublons.empty:
    display(doublons)
    raise ValueError("Des CODGEO sont dupliqués.")

if not df["REG"].eq("11").all():
    display(df.loc[~df["REG"].eq("11")])
    raise ValueError("Le fichier contient des territoires hors Île-de-France.")

departements_idf = {"75", "77", "78", "91", "92", "93", "94", "95"}
departements_inattendus = set(df["DEP"].dropna()) - departements_idf
if departements_inattendus:
    raise ValueError(f"Départements inattendus : {departements_inattendus}")

valeurs_manquantes = df[variables_demographiques].isna().sum()
valeurs_manquantes = valeurs_manquantes.loc[valeurs_manquantes.gt(0)]
if not valeurs_manquantes.empty:
    display(valeurs_manquantes.rename("nombre_de_valeurs_manquantes"))
    raise ValueError("Certaines variables démographiques sont incomplètes.")

if df["P22_POP"].le(0).any() or df["P16_POP"].le(0).any():
    display(df.loc[df["P22_POP"].le(0) | df["P16_POP"].le(0)])
    raise ValueError("Certaines populations totales sont nulles ou négatives.")

print(f"Contrôles réussis pour {len(df):,} communes.")

Contrôles réussis pour 1,266 communes.


## 6. Calcul des indicateurs démographiques

In [7]:
indicateurs = df.rename(
    columns={
        "P11_POP": "POPULATION_2011",
        "P16_POP": "POPULATION_2016",
        "P22_POP": "POPULATION_2022",
        "P22_POP0014": "POP_0_14_ANS_2022",
        "P22_POP1529": "POP_15_29_ANS_2022",
        "P22_POP3044": "POP_30_44_ANS_2022",
        "P22_POP4559": "POP_45_59_ANS_2022",
        "P22_POP6074": "POP_60_74_ANS_2022",
        "P22_POP7589": "POP_75_89_ANS_2022",
        "P22_POP90P": "POP_90_ANS_PLUS_2022",
    }
).copy()

indicateurs["POP_MOINS_30_ANS_2022"] = (
    indicateurs["POP_0_14_ANS_2022"] + indicateurs["POP_15_29_ANS_2022"]
)
indicateurs["POP_15_44_ANS_2022"] = (
    indicateurs["POP_15_29_ANS_2022"] + indicateurs["POP_30_44_ANS_2022"]
)
indicateurs["POP_60_ANS_PLUS_2022"] = (
    indicateurs["POP_60_74_ANS_2022"]
    + indicateurs["POP_75_89_ANS_2022"]
    + indicateurs["POP_90_ANS_PLUS_2022"]
)
indicateurs["POP_75_ANS_PLUS_2022"] = (
    indicateurs["POP_75_89_ANS_2022"] + indicateurs["POP_90_ANS_PLUS_2022"]
)

def calculer_part(numerateur):
    return numerateur / indicateurs["POPULATION_2022"] * 100

indicateurs["PART_0_14_ANS_PCT"] = calculer_part(indicateurs["POP_0_14_ANS_2022"])
indicateurs["PART_15_29_ANS_PCT"] = calculer_part(indicateurs["POP_15_29_ANS_2022"])
indicateurs["PART_30_44_ANS_PCT"] = calculer_part(indicateurs["POP_30_44_ANS_2022"])
indicateurs["PART_MOINS_30_ANS_PCT"] = calculer_part(indicateurs["POP_MOINS_30_ANS_2022"])
indicateurs["PART_15_44_ANS_PCT"] = calculer_part(indicateurs["POP_15_44_ANS_2022"])
indicateurs["PART_60_ANS_PLUS_PCT"] = calculer_part(indicateurs["POP_60_ANS_PLUS_2022"])
indicateurs["PART_75_ANS_PLUS_PCT"] = calculer_part(indicateurs["POP_75_ANS_PLUS_2022"])

indicateurs["EVOLUTION_POP_2016_2022"] = (
    indicateurs["POPULATION_2022"] - indicateurs["POPULATION_2016"]
)
indicateurs["EVOLUTION_POP_2016_2022_PCT"] = (
    indicateurs["POPULATION_2022"] / indicateurs["POPULATION_2016"] - 1
) * 100
indicateurs["TAUX_ANNUEL_POP_2016_2022_PCT"] = (
    (indicateurs["POPULATION_2022"] / indicateurs["POPULATION_2016"]) ** (1 / 6) - 1
) * 100

colonnes_pourcentage = [
    "PART_0_14_ANS_PCT",
    "PART_15_29_ANS_PCT",
    "PART_30_44_ANS_PCT",
    "PART_MOINS_30_ANS_PCT",
    "PART_15_44_ANS_PCT",
    "PART_60_ANS_PLUS_PCT",
    "PART_75_ANS_PLUS_PCT",
    "EVOLUTION_POP_2016_2022_PCT",
    "TAUX_ANNUEL_POP_2016_2022_PCT",
]

print("Indicateurs calculés.")
display(indicateurs.head())

Indicateurs calculés.


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508,513386.756029,457511.658564,387099.21535,302819.494109,154040.332162,24566.982278,787667.317537,970898.414593,481426.808549,178607.31444,12.976293,24.288477,21.64501,37.264771,45.933487,22.776443,8.449964,-76622.0,-3.498199,-0.591718
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135095,159.522993,189.869837,347.331906,204.113784,89.442063,10.584321,341.658088,349.39283,304.140169,100.026384,15.396035,13.484615,16.049859,28.88065,29.534474,25.709228,8.455316,44.0,3.863038,0.633715
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936214,122.351987,146.913593,151.603361,188.399317,59.320602,24.474926,250.288201,269.265581,272.194845,83.795528,15.582974,14.9028,17.894469,30.485774,32.797269,33.154061,10.20652,2.0,0.2442,0.040659
3,77003,Amponville,77,11,388.0,351.0,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0,101.0,110.0,90.0,28.0,17.948718,10.826211,20.512821,28.774929,31.339031,25.641026,7.977208,0.0,0.0,0.0
4,77004,Andrezel,77,11,299.0,285.0,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0,110.0,111.0,83.0,16.0,23.765432,10.185185,24.074074,33.950617,34.259259,25.617284,4.938272,39.0,13.684211,2.160582


## 7. Contrôles des indicateurs et des communes fusionnées

In [8]:
somme_classes_age = indicateurs[
    [
        "POP_0_14_ANS_2022",
        "POP_15_29_ANS_2022",
        "POP_30_44_ANS_2022",
        "POP_45_59_ANS_2022",
        "POP_60_74_ANS_2022",
        "POP_75_89_ANS_2022",
        "POP_90_ANS_PLUS_2022",
    ]
].sum(axis=1)

ecart_relatif_pct = (
    (somme_classes_age - indicateurs["POPULATION_2022"]).abs()
    / indicateurs["POPULATION_2022"]
    * 100
)

if ecart_relatif_pct.gt(0.01).any():
    lignes_en_ecart = indicateurs.loc[
        ecart_relatif_pct.gt(0.01),
        ["CODGEO", "NOM_COMMUNE", "POPULATION_2022"],
    ].copy()
    lignes_en_ecart["ECART_AGE_PCT"] = ecart_relatif_pct.loc[
        ecart_relatif_pct.gt(0.01)
    ]
    display(lignes_en_ecart)
    raise ValueError("Les classes d’âge ne correspondent pas à la population totale.")

for colonne in colonnes_pourcentage[:7]:
    hors_limites = ~indicateurs[colonne].between(0, 100, inclusive="both")
    if hors_limites.any():
        display(indicateurs.loc[hors_limites, ["CODGEO", "NOM_COMMUNE", colonne]])
        raise ValueError(f"Pourcentage hors limites dans {colonne}.")

colonnes_numeriques_finales = indicateurs.select_dtypes(include="number").columns
matrice_numerique = (
    indicateurs[colonnes_numeriques_finales]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float, na_value=np.nan)
)
if not np.isfinite(matrice_numerique).all():
    raise ValueError("Le tableau contient une valeur numérique infinie.")

if not indicateurs["CODGEO"].eq("93066").any():
    raise ValueError("Saint-Denis (93066) est absente.")

if indicateurs["CODGEO"].eq("93059").any():
    raise ValueError(
        "Pierrefitte-sur-Seine (93059) apparaît encore comme commune indépendante."
    )

controle_communes = indicateurs.loc[
    indicateurs["CODGEO"].isin(["93066", "95680"]),
    [
        "CODGEO",
        "NOM_COMMUNE",
        "DEP",
        "POPULATION_2022",
        "PART_MOINS_30_ANS_PCT",
        "EVOLUTION_POP_2016_2022_PCT",
    ],
]
display(controle_communes)
print("Tous les contrôles sont réussis.")

,CODGEO,NOM_COMMUNE,DEP,POPULATION_2022,PART_MOINS_30_ANS_PCT,EVOLUTION_POP_2016_2022_PCT
1027,93066,Saint-Denis,93,148907.0,44.477183,5.637059
1263,95680,Villiers-le-Bel,95,29238.0,46.897909,7.307226


Tous les contrôles sont réussis.


## 8. Aperçu des communes ayant la plus forte part de moins de 30 ans

Cet aperçu sert uniquement à vérifier que les résultats sont interprétables. Il ne constitue pas encore un classement commercial.

In [12]:
apercu_jeunesse = (
    indicateurs[
        [
            "CODGEO",
            "NOM_COMMUNE",
            "DEP",
            "POPULATION_2022",
            "PART_MOINS_30_ANS_PCT",
            "EVOLUTION_POP_2016_2022_PCT",
        ]
    ]
    .sort_values("PART_MOINS_30_ANS_PCT", ascending=False)
    .head(10)
)
display(apercu_jeunesse)

,CODGEO,NOM_COMMUNE,DEP,POPULATION_2022,PART_MOINS_30_ANS_PCT,EVOLUTION_POP_2016_2022_PCT
1247,95611,Theuville,95,53.0,52.830189,39.473684
837,91235,Fleury-Mérogis,91,13816.0,49.964914,20.874891
1119,95127,Cergy,95,69578.0,49.749299,9.02225
933,91602,Souzy-la-Briche,91,484.0,49.372933,15.513126
1004,93014,Clichy-sous-Bois,93,29551.0,48.980991,-0.951902
239,77251,Lieusaint,77,14096.0,48.756131,5.485295
849,91286,Grigny,91,26500.0,48.579388,-8.488155
1008,93030,Dugny,93,11723.0,48.373004,9.982175
1260,95675,Villeron,95,1672.0,48.043184,123.529412
628,78322,Jouy-en-Josas,78,7985.0,48.013363,-3.294175


## 9. Export du tableau J2

In [13]:
indicateurs_export = indicateurs.copy()
indicateurs_export[colonnes_pourcentage] = (
    indicateurs_export[colonnes_pourcentage].round(2)
)

DOSSIER_PROCESSED_INSEE.mkdir(parents=True, exist_ok=True)
indicateurs_export.to_csv(
    FICHIER_SORTIE,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

print("J2 terminé.")
print(f"Fichier créé : {FICHIER_SORTIE}")
print(f"Communes : {len(indicateurs_export):,}")
print(f"Colonnes : {len(indicateurs_export.columns):,}")

J2 terminé.
Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\indicateurs_demographiques_idf_2022.csv
Communes : 1,266
Colonnes : 28


## Fin du J2

Lorsque la dernière cellule affiche **J2 terminé** :

1. enregistre le notebook avec `Ctrl + S` ;
2. place-le dans `notebooks/02_indicateurs_demographiques_idf.ipynb` dans ton dépôt local ;
3. dans GitHub Desktop, utilise le résumé `Création des indicateurs démographiques IDF` ;
4. clique sur **Commit to main**, puis **Push origin**.

Le CSV produit reste dans OneDrive et ne doit pas être ajouté à GitHub.